In [1]:
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

In [2]:
import pandas as pd

cols = [
    "title",
    "abstract",
    "categories"
]

df = pd.read_parquet(
    "../data/processed/publication_dataset.parquet",
    columns=cols
)

print(df.shape)

(3107014, 3)


In [3]:
df["text"] = (
    df["title"].fillna("")
    + " "
    + df["abstract"].fillna("")
)

In [4]:
df["text"]

0          Calculation of prompt diphoton production cros...
1          Sparsity-certifying Graph Decompositions   We ...
2          The evolution of the Earth-Moon system based o...
3          A determinant of Stirling cycle numbers counts...
4          From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...
                                 ...                        
3107009    On the origin of the irreversibility line in t...
3107010    Nonlinear Response of HTSC Thin Film Microwave...
3107011    Critical State Flux Penetration and Linear Mic...
3107012    Density of States and NMR Relaxation Rate in A...
3107013    Ginzburg Landau theory for d-wave pairing and ...
Name: text, Length: 3107014, dtype: str

In [5]:
df["primary_category"] = (
    df["categories"]
    .str.split()
    .str[0]
)

In [6]:
sample_df = df.sample(
    n=300000,
    random_state=42
).reset_index(drop=True)

print(sample_df.shape)

ArrowMemoryError: malloc of size 301607680 failed

In [ ]:
del df

import gc

gc.collect()

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("ggplot")

In [2]:
df = pd.read_parquet(
    "../data/processed/publication_dataset.parquet",
    columns=["categories"]
)

print(df.shape)

(3107014, 1)


In [3]:
df["primary_category"] = (
    df["categories"]
    .fillna("Unknown")
    .str.split()
    .str[0]
)

df.head()

,categories,primary_category
0,hep-ph,hep-ph
1,math.CO cs.CG,math.CO
2,physics.gen-ph,physics.gen-ph
3,math.CO,math.CO
4,math.CA math.FA,math.CA


In [4]:
print("=" * 60)

print("Total Papers :", len(df))

print("Unique Categories :", df["primary_category"].nunique())

print("=" * 60)

Total Papers : 3107014
Unique Categories : 171


In [6]:
category_counts = (
    df["primary_category"]
    .value_counts()
)

category_counts.head(20)

primary_category
cs.CV                 155379
hep-ph                143717
cs.LG                 139745
quant-ph              133684
hep-th                113809
astro-ph               94246
cs.CL                  84653
gr-qc                  72395
cond-mat.mtrl-sci      71861
cond-mat.mes-hall      69993
math.AP                58586
astro-ph.GA            55647
math.CO                54467
cond-mat.str-el        53151
astro-ph.SR            48734
astro-ph.HE            46785
astro-ph.CO            45409
math.PR                44573
cond-mat.stat-mech     44092
cs.AI                  41797
Name: count, dtype: int64

In [7]:
category_counts.tail(20)

primary_category
stat.OT     799
cs.OS       709
q-bio.SC    700
nlin.CG     586
dg-ga       562
patt-sol    452
funct-an    320
adap-org    306
mtrl-th     165
comp-gas    140
chem-ph     129
cs.GL       123
supr-con     69
atom-ph      68
acc-phys     46
plasm-ph     28
ao-sci       13
bayes-an     11
cs.SY         2
q-fin.EC      1
Name: count, dtype: int64

In [8]:
print(category_counts.describe())

count       171.000000
mean      18169.672515
std       27414.701407
min           1.000000
25%        2407.000000
50%        7735.000000
75%       21355.000000
max      155379.000000
Name: count, dtype: float64


In [9]:
thresholds = [10, 50, 100, 500, 1000, 5000]

for t in thresholds:

    count = (category_counts < t).sum()

    print(f"Categories with less than {t} papers : {count}")

Categories with less than 10 papers : 2
Categories with less than 50 papers : 6
Categories with less than 100 papers : 8
Categories with less than 500 papers : 15
Categories with less than 1000 papers : 22
Categories with less than 5000 papers : 68


In [10]:
threshold = 1000

valid_categories = category_counts[
    category_counts >= threshold
]

print(f"Threshold : {threshold}")

print(f"Remaining Categories : {len(valid_categories)}")

Threshold : 1000
Remaining Categories : 149


In [12]:
filtered_df = df[
    df["primary_category"].isin(valid_categories.index)
]

print(filtered_df.shape)

(3100047, 2)


In [13]:
percentage = (
    len(filtered_df)
    / len(df)
) * 100

print(f"Remaining Papers : {percentage:.2f}%")

Remaining Papers : 99.78%
